In [1]:
import shutil
import os

if os.path.exists("logs"):
    shutil.rmtree("logs")
    print("Logs directory cleared")
else:
    print("No logs directory found")

Logs directory cleared


In [2]:
from utils import start_tensorboard

start_tensorboard()

⚠️  Warning: Logs directory not found at c:\Users\Phil\Dev\lecture-notes-CO2-2026\exercise_8\logs


# Introduction to Keras and Layer types
Previously you built a simple artificial neural net (ANN) with one hidden layer and sigmoid activation function. Now we will take a look at a few adaptations of such a basic net both in terms of NN architecture as well as layer types. We will still stay with Fully Connected Neural Networks for this first exercise!

things to go through:
- activation functions (issue with sigmoid -> ReLU, Leaky ReLU)
- drop out
- skip connections
- deeper neural nets
- potential issues of Fully Connected Neural Networks (Scaling?)
- Keras things like functional API

## Setup things
Make sure your environment has all the required packages available. Take care to have
- keras (Read [This short guide](https://keras.io/getting_started/))
- a backend of your choosing for Keras (I dont care which one you use but we will stay with Tensorflow for now)
- scikit-learn
- pandas
- tensorboard (optional)

Feel free to use a package manager of your choice (avoid conda) or a premade environment in Azure/Colab.
Also make sure you have the data files you need ready, I recommend putting them into a ./data/ subdir

## Loading and preparing the dataset


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction
df = pd.read_csv("data/energydata_complete.csv")

# For timeseries such as this, are rows really independent?
# Is there a separate approach to this problem? What patterns do we lose by treating the dataset like this?
df['date_time'] = pd.to_datetime(df['date'])

df['dayinweek'] = df['date_time'].dt.day_of_week
df['month'] = df['date_time'].dt.month
df['hour'] = df["date_time"].dt.hour
df['minutes'] = df['date_time'].dt.minute
df.drop(columns=['date_time'], inplace=True)

# how do we keep it random but reproducible?
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["Appliances","date"]), df['Appliances'],random_state=42)
print(X_train.shape, X_test.shape)

sd = StandardScaler()
X_train_scaled = sd.fit_transform(X_train)
X_test_scaled = sd.transform(X_test)
X_train.head(3)

(14801, 31) (4934, 31)


,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,...,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2,dayinweek,month,hour,minutes
8242,0,21.00,38.163333,18.50000,40.326667,20.230,38.00,19.926667,34.526667,18.730000,...,75.333333,4.666667,40.000000,0.600000,6.455464,6.455464,1,3,22,40
10603,0,21.00,40.450000,18.39000,44.090000,22.000,38.29,19.856667,38.863333,19.323333,...,97.000000,6.000000,50.833333,5.916667,6.797277,6.797277,4,3,8,10
18910,0,24.69,49.476000,24.32973,47.159009,26.834,43.84,23.997297,46.901351,22.927027,...,82.333333,2.000000,40.000000,14.433333,8.524160,8.524160,6,5,0,40


## Vanishing Gradient Problem with Sigmoid

Sigmoid's derivative ranges from 0 to 0.25 (max at the middle).

During backpropagation, gradients multiply layer-by-layer: 0.25 × 0.25 × 0.25...

**Result:** Gradients shrink exponentially in deeper networks → early layers barely learn. **Deeper Networks fail with sigmoid**

**Solution:** Use e.g. ReLU (derivative = 1 for positive inputs).

In [4]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print(f"Using Keras {keras.__version__} with backend: {keras.backend.backend()}")

Using Keras 3.12.0 with backend: tensorflow


In [5]:
import keras
from utils import train_model,eval_model

model_sigmoid = keras.models.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(512, activation="sigmoid"),
    keras.layers.Dense(256, activation="sigmoid"),
    keras.layers.Dense(128, activation="sigmoid"),
    keras.layers.Dense(64, activation="sigmoid"),
    keras.layers.Dense(32, activation="sigmoid"),
    keras.layers.Dense(1, activation=None)
])

model_sigmoid.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_sigmoid.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model_relu = keras.models.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation=None)
])

model_relu.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_relu.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history_sigmoid = train_model(model_sigmoid, X_train_scaled, y_train, "sigmoid_deep")

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 18987.2949 - mae: 90.6759 - val_loss: 17808.2695 - val_mae: 85.6665
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 17491.1914 - mae: 82.0303 - val_loss: 16521.6094 - val_mae: 77.7950
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 16307.2139 - mae: 74.5734 - val_loss: 15437.2275 - val_mae: 70.7072
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 15295.0352 - mae: 67.7636 - val_loss: 14502.5625 - val_mae: 64.2280
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 14426.7822 - mae: 61.6727 - val_loss: 13705.9307 - val_mae: 58.4311
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 13682.0205 - mae: 56.7650 - val_loss: 13021.8613 - val_mae: 54.0110
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 13052.6592 - mae: 52.8735 - val_loss: 12451.3633 - val_mae: 51.2635
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12524.4053 - mae: 51.1699 - val_loss: 11972.

In [8]:
history_relu = train_model(model_relu, X_train_scaled, y_train, "relu_deep")

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 10180.5469 - mae: 57.0170 - val_loss: 8522.1914 - val_mae: 50.7680
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8840.6514 - mae: 51.8195 - val_loss: 8352.6055 - val_mae: 58.4020
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8418.5732 - mae: 50.2890 - val_loss: 7885.4570 - val_mae: 48.2014
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8140.0718 - mae: 48.6533 - val_loss: 8000.6250 - val_mae: 52.5693
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7951.0718 - mae: 48.2340 - val_loss: 7572.5645 - val_mae: 46.0352
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7614.9351 - mae: 46.5292 - val_loss: 7470.8296 - val_mae: 50.8094
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7423.2090 - mae: 45.7280 - val_loss: 7499.6006 - val_mae: 50.4672
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7259.3745 - mae: 45.3333 - val_loss: 7705.7793 - val_mae:

In [9]:
model_leaky_relu = keras.models.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)), #
    keras.layers.Dense(512),
    keras.layers.LeakyReLU(negative_slope=0.01),
    keras.layers.Dense(256),
    keras.layers.LeakyReLU(negative_slope=0.01),
    keras.layers.Dense(128),
    keras.layers.LeakyReLU(negative_slope=0.01),
    keras.layers.Dense(64),
    keras.layers.LeakyReLU(negative_slope=0.01),
    keras.layers.Dense(32),
    keras.layers.LeakyReLU(negative_slope=0.01),
    keras.layers.Dense(1, activation=None)
])

model_leaky_relu.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_leaky_relu.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history_sigmoid = train_model(model_sigmoid, X_train_scaled, y_train, "sigmoid_deep")
history_relu = train_model(model_relu, X_train_scaled, y_train, "relu_deep")
history_leaky_relu = train_model(model_leaky_relu, X_train_scaled, y_train, "leaky_relu_deep")

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.4443 - mae: 60.9841 - val_loss: 10469.5908 - val_mae: 60.3302
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.5244 - mae: 61.0312 - val_loss: 10469.5566 - val_mae: 60.2891
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.4043 - mae: 61.0110 - val_loss: 10469.5762 - val_mae: 60.3149
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.3564 - mae: 60.9739 - val_loss: 10469.6240 - val_mae: 60.3596
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.3955 - mae: 61.0475 - val_loss: 10469.6436 - val_mae: 60.3737
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.5088 - mae: 60.9588 - val_loss: 10469.7314 - val_mae: 60.4283
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.4824 - mae: 61.0673 - val_loss: 10469.6924 - val_mae: 60.4061
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 10761.4326 - mae: 61.0458 - val_loss: 10469.

In [11]:
# eval on test set
from utils import eval_model

eval_model(model_sigmoid, X_test_scaled, y_test, "Sigmoid Deep Network")
eval_model(model_relu, X_test_scaled, y_test, "ReLU Deep Network")
eval_model(model_leaky_relu, X_test_scaled, y_test, "Leaky ReLU Deep Network")


  Sigmoid Deep Network - Test Results
  Test Loss (MSE): 9,935.09
  Test MAE:        59.95
  R² Score:        -0.0001 (-0.01% variance explained)


  ReLU Deep Network - Test Results
  Test Loss (MSE): 6,476.61
  Test MAE:        39.47
  R² Score:        0.3480 (34.80% variance explained)


  Leaky ReLU Deep Network - Test Results
  Test Loss (MSE): 6,527.34
  Test MAE:        41.38
  R² Score:        0.3429 (34.29% variance explained)



(6527.33642578125, 41.378780364990234, 0.3429204821586609)

## Dropout - Preventing Overfitting

Deep networks can **memorize trained-on data** instead of learning generalizations.

**Dropout** randomly sets a percentage of neurons to 0 during each training step (e.g., 30% dropout → 30% of neurons turned off).

This forces the network to learn **robust features** that don't rely on a specific small subset of neurons. We will see on the tensorboard curves, that the model is learning slower but also less likely to overfit.

In [12]:
## Dropout 
model_with_dropout = keras.models.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation=None)
])

model_with_dropout.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_with_dropout.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history_with_dropout = train_model(model_with_dropout, X_train_scaled, y_train, "with_dropout", epochs=50)

Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 10716.2266 - mae: 57.7603 - val_loss: 8814.8682 - val_mae: 47.1681
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 9490.3027 - mae: 53.4467 - val_loss: 8371.8633 - val_mae: 48.7251
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 9158.5889 - mae: 52.8593 - val_loss: 8499.7285 - val_mae: 46.3610
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8943.5205 - mae: 51.3906 - val_loss: 8397.4072 - val_mae: 43.7438
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8769.9570 - mae: 50.5846 - val_loss: 7722.8193 - val_mae: 45.1725
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8543.2949 - mae: 49.5896 - val_loss: 7732.5229 - val_mae: 43.1921
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8475.6191 - mae: 49.2469 - val_loss: 7922.7539 - val_mae: 46.5302
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8253.3408 - mae: 48.3603 - val_loss: 7834.7158 - val_mae:

In [14]:
eval_model(model_with_dropout, X_test_scaled, y_test, "ReLU with Dropout")


  ReLU with Dropout - Test Results
  Test Loss (MSE): 6,654.77
  Test MAE:        37.91
  R² Score:        0.3301 (33.01% variance explained)



(6654.76513671875, 37.905738830566406, 0.33009278774261475)

# Batchnorm
**Problem:** During training, the distribution of inputs to each layer changes as previous layers update (internal covariate shift), slowing down training.

**Batch Normalization** normalizes layer inputs by computing mean and standard deviation across the batch, then scaling and shifting with learned parameters.

Batchnorm is quite optional, at the end the effect on performance should be tested empirically. 


In [15]:
model_with_batchnorm = keras.models.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(512),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation=None)
])

model_with_batchnorm.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_with_batchnorm.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 194,945 (761.50 KB)

 Trainable params: 192,961 (753.75 KB)

 Non-trainable params: 1,984 (7.75 KB)

In [16]:
history_with_batchnorm = train_model(model_with_batchnorm, X_train_scaled, y_train, "with_dropout_and_batchnorm", epochs=50)

Epoch 1/50


370/370 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 19244.8926 - mae: 93.9951 - val_loss: 17348.5254 - val_mae: 89.0292
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 15644.1201 - mae: 78.8735 - val_loss: 13014.2793 - val_mae: 69.1179
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 11666.7822 - mae: 60.9490 - val_loss: 9649.7812 - val_mae: 54.7968
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9360.8115 - mae: 50.8664 - val_loss: 8320.4932 - val_mae: 45.8680
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8602.4521 - mae: 48.8080 - val_loss: 7408.5845 - val_mae: 43.8128
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8278.8633 - mae: 48.4592 - val_loss: 7419.5361 - val_mae: 43.1932
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8172.0181 - mae: 48.8634 - val_loss: 7438.6313 - val_mae: 43.2855
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8171.6455 - mae: 48.3958 - val_loss: 7393.2832 - val_mae: 44.483

In [17]:
eval_model(model_with_batchnorm, X_test_scaled, y_test, "ReLU with Dropout and Batchnorm")


  ReLU with Dropout and Batchnorm - Test Results
  Test Loss (MSE): 6,328.30
  Test MAE:        39.19
  R² Score:        0.3630 (36.30% variance explained)



(6328.30078125, 39.18722152709961, 0.3629564046859741)


## Keras Functional API

The **Sequential API** is simple but limited - layers stack linearly, no branching or multiple inputs/outputs. It is quite rigid in how we can define and interact with the architecture.

The **Functional API** is more flexible (and follows typical functional programming styles - think lambda):
- Multiple inputs/outputs
- Layer sharing
- Branching and merging
- Skip connections (residual networks)
- also it allows us to dynamically construct different architectures e.g. for hyperparam search.


In [18]:
# Functional API version of dropout model
inputs = keras.layers.Input(shape=(X_train_scaled.shape[1],))

x = keras.layers.Dense(512, activation="relu")(inputs)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(1, activation=None)(x)

model_functional = keras.Model(inputs=inputs, outputs=outputs, name="functional_dropout")
model_functional.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_functional.summary()

Model: "functional_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 31)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 512)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 190,977 (746.00 KB)

 Trainable params: 190,977 (746.00 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Your turn, implement a skip connection from the first dense+dropout layer block to the last (excluding output layer)
#Hint https://keras.io/api/layers/merging_layers/add/

In [20]:
inputs = keras.layers.Input(shape=(X_train_scaled.shape[1],))

x = keras.layers.Dense(256, activation="relu")(inputs)
x = keras.layers.Dropout(0.3)(x)
skip = x

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)

x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

x = keras.layers.Add()([x, skip])

x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(1, activation=None)(x)

model_skip = keras.Model(inputs=inputs, outputs=outputs, name="skip_connection")
model_skip.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_skip.summary()


Model: "skip_connection"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 31)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_36 (Dense)    │ (None, 256)       │      8,192 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 256)       │          0 │ dense_36[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_37 (Dense)    │ (None, 256)       │     65,792 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 256)       │          0 │ dense_37[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_38 (Dense)    │ (None, 256)       │     65,792 │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 256)       │          0 │ dense_38[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_39 (Dense)    │ (None, 256)       │     65,792 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 256)       │          0 │ dense_39[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256)       │          0 │ dropout_18[0][0], │
│                     │                   │            │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_40 (Dense)    │ (None, 64)        │     16,448 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 64)        │          0 │ dense_40[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_41 (Dense)    │ (None, 1)         │         65 │ dropout_19[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 222,081 (867.50 KB)

 Trainable params: 222,081 (867.50 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
history_skip = train_model(model_skip, X_train_scaled, y_train, "with_skip_connection", epochs=50)


Epoch 1/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 10301.2441 - mae: 56.4873 - val_loss: 9110.8467 - val_mae: 45.4563
Epoch 2/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 9226.6279 - mae: 52.7984 - val_loss: 8075.2725 - val_mae: 47.8956
Epoch 3/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8937.4072 - mae: 51.1913 - val_loss: 7769.0889 - val_mae: 46.0928
Epoch 4/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8557.5664 - mae: 49.7242 - val_loss: 7552.3521 - val_mae: 46.9988
Epoch 5/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8409.3369 - mae: 49.0125 - val_loss: 8064.3774 - val_mae: 41.8275
Epoch 6/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8215.9502 - mae: 47.9341 - val_loss: 7715.4556 - val_mae: 42.3934
Epoch 7/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7919.9087 - mae: 47.1441 - val_loss: 8175.7891 - val_mae: 42.8510
Epoch 8/50
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7797.1318 - mae: 46.9508 - val_loss: 7434.6680 - val_mae:

In [22]:
eval_model(model_skip, X_test_scaled, y_test, "Skip Connection Network")



  Skip Connection Network - Test Results
  Test Loss (MSE): 6,345.23
  Test MAE:        38.33
  R² Score:        0.3613 (36.13% variance explained)



(6345.228515625, 38.33418655395508, 0.3612525463104248)